# BoltzGen on Colab — free, hotspot-targeted binder design

**Why BoltzGen and not BoltzProt.** BoltzProt-1 is API-only — the weights are not published, so it cannot be self-hosted at any price. BoltzGen is its generative core, MIT-licensed, weights on Hugging Face. You run it here on Colab's GPU and pay nothing but compute units.

What you give up is **BoltzPPI**, the ranker that took BoltzProt's nanobody hit rate from 3.3% to 8.0%. That gain came from *ranking*, not generation — so the sane plan is: generate free here, then rank with your own Boltz-2 / meta-scorer stack downstream.

**This run is different from an API run: the compute is local, so a dropped session DOES kill it.** Everything below is therefore written in resumable batches — each batch's output lands on Drive the moment it finishes, and re-running skips whatever is already done.

### Setup
1. **Runtime → Change runtime type → GPU** (A100 if you have it; T4 works and is far cheaper per unit).
2. Colab Pro+ **background execution** if you want it to survive closing the laptop.
3. Weights (~6 GB) cache to Drive on first run, so later sessions skip the download.

In [ ]:
#@title 1. GPU check + Drive + install
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — switch runtime type to GPU'

from google.colab import drive
drive.mount('/content/drive')

import os
# Cache weights on Drive so the 6GB download happens once, not once per session.
os.environ['HF_HOME'] = '/content/drive/MyDrive/boltzgen/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

!pip -q install boltzgen
print('\nboltzgen installed; HF_HOME =', os.environ['HF_HOME'])

In [ ]:
#@title 2. Config — edit these
from pathlib import Path

RUN_DIR    = Path('/content/drive/MyDrive/boltzgen/5g51_run01')
TARGET_CIF = Path('/content/drive/MyDrive/boltzgen/targets/5G51_ext_seq60.cif')

TARGET_CHAIN   = 'B'            # chain B of that CIF is the 5G51 target
HOTSPOTS       = '146,149,152'  # I405 / F408 / I411 in this file's numbering
BINDER_LENGTH  = '80..140'      # sampled per design, inclusive
PROTOCOL       = 'protein-anything'

# Resumable batching. Total designs = N_BATCHES * DESIGNS_PER_BATCH.
# Small batches = more checkpoints = less lost to a disconnect.
N_BATCHES         = 10
DESIGNS_PER_BATCH = 50
BUDGET_PER_BATCH  = 10          # diversity-optimised keepers per batch

RUN_DIR.mkdir(parents=True, exist_ok=True)
assert TARGET_CIF.exists(), f'Target CIF not found: {TARGET_CIF}'
print('Run dir:', RUN_DIR)

In [ ]:
#@title 3. Build the YAML spec (this is where hotspots get enforced)
import shutil, yaml

# BoltzGen resolves `file.path` relative to the YAML, so keep them together.
spec_dir = RUN_DIR / 'spec'
spec_dir.mkdir(parents=True, exist_ok=True)
local_cif = spec_dir / TARGET_CIF.name
if not local_cif.exists():
    shutil.copy(TARGET_CIF, local_cif)

spec = {
    'entities': [
        {'protein': {'id': 'C', 'sequence': BINDER_LENGTH}},
        {'file': {'path': local_cif.name,
                  'include': [{'chain': {'id': TARGET_CHAIN}}]}},
    ],
    # The whole point. V4 kept missing F408 because nothing forced it to bind
    # there; binding_types makes the epitope an explicit constraint instead of
    # something you hope for and filter on afterwards.
    'binding_types': [
        {'chain': {'id': TARGET_CHAIN, 'binding': HOTSPOTS}},
    ],
}

spec_path = spec_dir / 'design.yaml'
with open(spec_path, 'w') as f:
    yaml.safe_dump(spec, f, sort_keys=False)

print(spec_path.read_text())

In [ ]:
#@title 4. Smoke test — 2 designs. Do NOT skip this.
# A 10-batch run that dies on a config error in batch 1 wastes an hour of GPU.
smoke = RUN_DIR / '_smoke'
!boltzgen run "{spec_path}" --output "{smoke}" --protocol {PROTOCOL} --num_designs 2 --budget 1

import glob
hits = glob.glob(str(smoke / '**' / '*.csv'), recursive=True)
print('\nCSVs produced:', hits if hits else 'NONE — fix before continuing')

In [ ]:
#@title 5. Batched run — interrupt and re-run freely
import json, os, tempfile, time, subprocess
from datetime import datetime, timezone

STATE = RUN_DIR / 'batches.json'

def _now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def load_state():
    return json.load(open(STATE)) if STATE.exists() else {}

def save_state(s):
    """Atomic: a truncated state file would make the run restart from zero."""
    fd, tmp = tempfile.mkstemp(dir=str(RUN_DIR), suffix='.tmp')
    try:
        with os.fdopen(fd, 'w') as f:
            json.dump(s, f, indent=1); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, STATE)
    except BaseException:
        if os.path.exists(tmp): os.unlink(tmp)
        raise

state = load_state()
print(f'{sum(1 for v in state.values() if v.get("status")=="done")} batch(es) already complete\n')

for i in range(N_BATCHES):
    key = f'batch_{i:03d}'
    if state.get(key, {}).get('status') == 'done':
        print(f'skip {key} — done'); continue

    out = RUN_DIR / key
    # A partial dir from a killed session would poison this batch's results.
    if out.exists():
        shutil.rmtree(out)

    state[key] = {'status': 'running', 'started_at': _now()}
    save_state(state)

    cmd = ['boltzgen', 'run', str(spec_path), '--output', str(out),
           '--protocol', PROTOCOL,
           '--num_designs', str(DESIGNS_PER_BATCH),
           '--budget', str(BUDGET_PER_BATCH)]
    print(f'--- {key}: {" ".join(cmd)}')
    t0 = time.time()
    rc = subprocess.run(cmd).returncode
    mins = (time.time() - t0) / 60

    if rc == 0:
        state[key] = {'status': 'done', 'minutes': round(mins, 1),
                      'finished_at': _now()}
        print(f'    done in {mins:.1f} min')
    else:
        state[key] = {'status': 'failed', 'returncode': rc,
                      'minutes': round(mins, 1), 'finished_at': _now()}
        print(f'    FAILED rc={rc} after {mins:.1f} min')
    save_state(state)

print('\nBatch status:', {k: v['status'] for k, v in sorted(state.items())})

In [ ]:
#@title 6. Merge every batch's metrics into one CSV
import pandas as pd, glob

frames = []
for p in sorted(glob.glob(str(RUN_DIR / 'batch_*' / '**' / 'all_designs_metrics.csv'),
                          recursive=True)):
    df = pd.read_csv(p)
    df['batch'] = Path(p).parts[len(RUN_DIR.parts)]
    df['source_csv'] = p
    frames.append(df)

if frames:
    allm = pd.concat(frames, ignore_index=True)
    out = RUN_DIR / 'all_batches_metrics.csv'
    allm.to_csv(out, index=False)
    print(f'{len(allm)} designs across {len(frames)} batches -> {out}')
    print('\nColumns:', list(allm.columns))
    display(allm.head(10))
else:
    print('No metrics CSVs found yet.')

## If the session dies

Run all again. Cell 5 reads `batches.json` from Drive and skips completed batches. You lose at most the one batch that was mid-flight — which is why `DESIGNS_PER_BATCH` is small. Lower it further if you're on a flaky connection.

## Unit burn

Unlike the API notebook, this one **needs** the GPU, so units burn the whole time (~2/hr on T4, ~12-15/hr on A100). Time the smoke test, multiply out, and check it against your 1200 before launching all 10 batches.

## Next step after this

These designs are ranked by BoltzGen's own confidence, which is exactly the weak ranking signal BoltzProt replaced. Push `all_batches_metrics.csv` through your Boltz-2 / ipSAE stack and your meta-scorer before ordering anything — and verify hotspot contacts in ChimeraX rather than trusting the score, the way we did for `ext_seq60`.